In [1]:

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor

sys.path.append('../')
sys.path.append('./')
import importlib
import yaml
import torch

from tqdm.auto import tqdm
import logging

# logging.getLogger('sox').setLevel(logging.ERROR)
# logger = logging.getLogger('sox')
# logger.setLevel('CRITICAL')


In [2]:
import lightning_scripts.lightning_ssl_matched_speech_in_noise as lightning 
importlib.reload(lightning)


LitAudioSSL = lightning.LitAudioSSL

## init config. Will be yaml eventually, but start as dict 
config_path = "model_configs/barlow_word_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 4
config['hparas']['batch_size'] = 64
config['hparas']['global_batch_size'] = 64
config['num_gpus'] = 1 

model = LitAudioSSL(config)


In [3]:
model = model.eval()

In [4]:
x = torch.randn(1,1,40000)

feature, out, logits = model.model(x, with_latent=True)

In [6]:
{layer: np.prod(tensor.shape).item() for layer, tensor in logits.items()}

{'input_after_preproc': 82290,
 'batchnorm0': 82290,
 'conv0': 886080,
 'relu0': 886080,
 'maxpool0': 224640,
 'batchnorm1': 224640,
 'conv1': 152064,
 'relu1': 152064,
 'maxpool1': 39168,
 'batchnorm2': 39168,
 'conv2': 78336,
 'relu2': 78336,
 'conv3': 156672,
 'relu3': 156672,
 'conv4': 78336,
 'relu4': 78336,
 'avgpool': 23040,
 'xview': 23040,
 'fullyconnected': 4096,
 'relufc': 4096,
 'dropout': 4096,
 'final': 4096}

In [4]:
train_ds = model.train_dataloader()

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [3]:
from lightning_scripts import jsinV3DataLoader_precombined_batched 

importlib.reload(jsinV3DataLoader_precombined_batched)
MatchedSpeechInNoiseDatasetBatched = jsinV3DataLoader_precombined_batched.MatchedSpeechInNoiseDatasetBatched

dataset = MatchedSpeechInNoiseDatasetBatched(speech_h5_path=config['data']['val_speech_h5_path'],
                                                     noise_h5_path=config['data']['val_noise_h5_path'],
                                                     low_db=config['audio_transforms']['low_snr'],
                                                     high_db=config['audio_transforms']['high_snr'],
                                                     db_spl=config['audio_transforms']['dbspl'],
                                                     batch_size=config['hparas']['batch_size'],
                                                     signal_augment=config['data'].get("signal_augment", False),
                                                     target_keys=config['data'].get("target_keys", None),
                                                     )
dataset[0]

For this stretch factor, the stretch effect has better performance.
For this stretch factor, the stretch effect has better performance.


([tensor([[-0.0012,  0.0005,  0.0075,  ...,  0.0083,  0.0114,  0.0106],
          [-0.0086, -0.0096, -0.0114,  ...,  0.0034,  0.0023,  0.0010],
          [-0.0196, -0.0218, -0.0119,  ..., -0.0124, -0.0126, -0.0129],
          ...,
          [-0.0038,  0.0314,  0.0240,  ..., -0.0089, -0.0148, -0.0009],
          [-0.0197, -0.0156, -0.0142,  ...,  0.0012,  0.0005, -0.0056],
          [-0.0081, -0.0045,  0.0013,  ...,  0.0002,  0.0020,  0.0042]]),
  tensor([[-0.0088, -0.0110, -0.0115,  ..., -0.0746, -0.0662, -0.0471],
          [ 0.0107,  0.0086,  0.0052,  ..., -0.0048, -0.0116, -0.0120],
          [-0.0052, -0.0048, -0.0053,  ...,  0.0042,  0.0009, -0.0037],
          ...,
          [-0.0128, -0.0212, -0.0366,  ...,  0.0032, -0.0058, -0.0159],
          [-0.0223, -0.0219, -0.0211,  ..., -0.0374, -0.0354, -0.0363],
          [-0.0105, -0.0108, -0.0117,  ..., -0.0099, -0.0090, -0.0083]]),
  tensor([[-2.2271e-03, -5.6190e-03, -7.7411e-03,  ..., -3.7424e-03,
            2.7225e-04, -1.1608e-

In [6]:
batch = dataset[0]

In [8]:
batch[0][0]

tensor([[ 5.4993e-03,  1.6702e-03, -1.0030e-02,  ..., -3.2607e-03,
         -1.3480e-02,  5.6814e-03],
        [-3.9527e-03, -5.4219e-03, -7.3263e-03,  ..., -4.8766e-02,
         -4.0447e-02, -2.2994e-02],
        [ 1.8882e-02,  2.8544e-02,  3.5556e-02,  ..., -1.8424e-03,
         -2.2241e-03, -4.0600e-03],
        ...,
        [ 2.3297e-05,  1.3476e-03,  3.9609e-03,  ..., -4.1165e-04,
         -3.9116e-03, -4.6539e-03],
        [ 1.2179e-02,  1.2813e-02,  1.0281e-02,  ..., -1.8667e-02,
         -2.1732e-02, -2.2851e-02],
        [-1.9480e-02, -1.8370e-02, -1.6020e-02,  ...,  5.9041e-03,
          2.5728e-03,  2.4078e-03]])

In [1]:
model

NameError: name 'model' is not defined

In [4]:
trainer = L.Trainer(
                    # callbacks=[lr_monitor],
                    # limit_train_batches=5,
                    limit_val_batches=2,
                    max_epochs=5,
                    # callbacks=callbacks,
                    #  strategy='ddp_notebook',
                    #  reload_dataloaders_every_n_epochs=-1,
                    devices=1)

trainer.fit(model)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name            | Type                       | Params | Mode 
-----------------------------------------------------------------------
0 | audio_rep       | AudioToAudioRepresentation | 0      | train
1 | model           | ModelWithFrontEnd          | 116 M  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Rank 0 N training batches 317
This is the first step after restoring from a checkpoint!


Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined